# FPS, задержка и память моделей

Сравнение включённых моделей через общий `Predictor`: CPU/CUDA, batch=1,
прогрев и отдельные процессы. [Подробный протокол и варианты измерений](scripts/benchmark/README.md).
Синтетический поток не подтверждает качество реального видео. Полный конвейер
защиты растений и HSV здесь не измеряется. Загрузка весов и обучение не запускаются.


In [4]:
from pathlib import Path
import sys

# Поддерживаются запуск из корня репозитория и из manual_notebooks.
NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / "scripts" / "common.py").is_file() else Path.cwd() / "manual_notebooks"
if not (NOTEBOOK_DIR / "scripts" / "common.py").is_file():
    raise RuntimeError("Open this notebook from the repository root or manual_notebooks")
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from scripts import common as cmn
from scripts.workflows import enabled_specs
from scripts.benchmark import BenchmarkConfig, run_benchmarks
from scripts.benchmark.plots import summary_table, plot_benchmarks, plot_fps_over_time, plot_latency_over_time


## Конфигурация

`cmn.MODELS` остаётся единственным источником выбора. Для изменения набора задайте,
например, `cmn.MODELS["rf-detr-m"] = False` перед созданием `CHECKPOINTS`.
Для воспроизводимого сравнения замените пути удобных копий на веса конкретных запусков.
Изображения читаются до измерения, затем циклически используются в заданном порядке.


In [5]:
# Измените настройки перед запуском. Недоступные файлы вызывают явную ошибку.
CHECKPOINTS = {
    spec.id: cmn.TRAINED_WEIGHTS_DIR / f"{spec.id}.pt"
    for spec in enabled_specs()
}
IMAGE_PATHS = [cmn.DATASET_PATH / "images" / "image_000646.png"]
DEVICES = ["cpu"]  # Например: ["cpu", "cuda:0"].
PREDICTION_OPTIONS = {
    spec.id: ({"imgsz": 640, "conf": 0.25, "rect": True}
              if spec.backend == "ultralytics" else {"conf": 0.25})
    for spec in enabled_specs()
}
# RF-DETR использует разрешение checkpoint; несовместимый imgsz вызовет ошибку.
PROMPTS = {}  # JSON-совместимые подсказки для моделей, которым они нужны.
RUN_NOTES = ""  # Оборудование, режим питания, внешние ограничения.

WARMUP_FRAMES = 30
MEASUREMENT_SECONDS = 30
INPUT_RATES = (30,)
ORDER_SEED = 20261028
MEMORY_INTERVAL_SECONDS = 0.01
OUTPUT_ROOT = cmn.ROOT / "experiments" / "benchmarks"

for spec in enabled_specs():
    print(spec.id, spec.backend, CHECKPOINTS[spec.id], spec.unavailable_reason("predict") or "Ready for preflight")


yolov8n ultralytics /home/natsusa/academic/tobacco-article/weights/trained/yolov8n.pt Ready for preflight
yolo26n ultralytics /home/natsusa/academic/tobacco-article/weights/trained/yolo26n.pt Ready for preflight


## Прямое измерение

Запустите следующую ячейку, когда нужно измерить direct inference.
Поток запускается собственной ячейкой ниже. Флагов включения нет.
Длительность direct: `MEASUREMENT_SECONDS` на пару модель/устройство
плюс загрузка и прогрев. Для smoke test: 2 кадра прогрева, 1 секунда.

Прогрев выполняется перед каждым окном с теми же параметрами. Каждый завершённый
замер сразу сохраняется вместе с задержками, памятью и окружением.
Путь к каталогу и worker.log выводится до начала работы.


In [6]:
config = BenchmarkConfig(
    warmup_frames=WARMUP_FRAMES, duration_s=MEASUREMENT_SECONDS,
    input_rates=INPUT_RATES,
    direct=True, stream=False,
    seed=ORDER_SEED, memory_interval_s=MEMORY_INTERVAL_SECONDS,
)
DIRECT_RESULT_DIRECTORY = run_benchmarks(
    CHECKPOINTS, IMAGE_PATHS, devices=DEVICES, config=config,
    prediction_options=PREDICTION_OPTIONS, prompts=PROMPTS,
    output_root=OUTPUT_ROOT, notes=RUN_NOTES,
)
RESULT_DIRECTORY = DIRECT_RESULT_DIRECTORY
print(f"Completed: {RESULT_DIRECTORY}")


Benchmark directory: /home/natsusa/academic/tobacco-article/experiments/benchmarks/20261121T092221Z-7c55a30a
Running yolov8n on cpu; log: /home/natsusa/academic/tobacco-article/experiments/benchmarks/20261121T092221Z-7c55a30a/yolov8n-cpu/worker.log
Running yolo26n on cpu; log: /home/natsusa/academic/tobacco-article/experiments/benchmarks/20261121T092221Z-7c55a30a/yolo26n-cpu/worker.log
Completed: /home/natsusa/academic/tobacco-article/experiments/benchmarks/20261121T092221Z-7c55a30a


## Синтетический поток

Запустите эту ячейку для sweep по `INPUT_RATES`; direct-ячейку запускать не обязательно.
Длительность: `MEASUREMENT_SECONDS × len(INPUT_RATES)` на пару
модель/устройство плюс загрузка и прогрев. При запуске всех ячеек выполняются оба режима.


In [7]:
config = BenchmarkConfig(
    warmup_frames=WARMUP_FRAMES, duration_s=MEASUREMENT_SECONDS,
    input_rates=INPUT_RATES,
    direct=False, stream=True,
    seed=ORDER_SEED, memory_interval_s=MEMORY_INTERVAL_SECONDS,
)
STREAM_RESULT_DIRECTORY = run_benchmarks(
    CHECKPOINTS, IMAGE_PATHS, devices=DEVICES, config=config,
    prediction_options=PREDICTION_OPTIONS, prompts=PROMPTS,
    output_root=OUTPUT_ROOT, notes=RUN_NOTES,
)
RESULT_DIRECTORY = STREAM_RESULT_DIRECTORY
print(f"Completed: {RESULT_DIRECTORY}")


Benchmark directory: /home/natsusa/academic/tobacco-article/experiments/benchmarks/20261121T092329Z-642d196c
Running yolov8n on cpu; log: /home/natsusa/academic/tobacco-article/experiments/benchmarks/20261121T092329Z-642d196c/yolov8n-cpu/worker.log
Running yolo26n on cpu; log: /home/natsusa/academic/tobacco-article/experiments/benchmarks/20261121T092329Z-642d196c/yolo26n-cpu/worker.log
Completed: /home/natsusa/academic/tobacco-article/experiments/benchmarks/20261121T092329Z-642d196c


## Таблица и графики без повторного inference

Задайте `SAVED_RESULT_DIRECTORY`, чтобы открыть прошлый или прерванный запуск.
По умолчанию отображается последний выполненный режим. Можно указать
`DIRECT_RESULT_DIRECTORY` или `STREAM_RESULT_DIRECTORY` для выбора режима.
Processing FPS — число завершённых кадров / фактическая длительность окна.
Frame latency включает копирование и очередь, поэтому её обратная величина не FPS.

RSS — память всего worker, выборочный пик с интервалом 10 мс. CUDA allocated/reserved —
отдельные показатели аллокатора PyTorch после прогрева, не полная VRAM устройства.
Их нельзя складывать с RAM или друг с другом. На CPU CUDA-поля отсутствуют (`null`).
Каждый режим и входная частота измеряются один раз на пару модель/устройство.


In [ ]:
SAVED_RESULT_DIRECTORY = None  # Например: OUTPUT_ROOT / "20261121T120000Z-..."
results_path = SAVED_RESULT_DIRECTORY or globals().get("RESULT_DIRECTORY")
if results_path is not None:
    from IPython.display import display
    display(summary_table(results_path))
    plot_benchmarks(results_path)
else:
    print("Run the benchmark or set SAVED_RESULT_DIRECTORY to an existing results directory.")


## FPS во времени для каждой модели

Отдельная фигура для каждой пары модель/устройство, отдельная панель для каждого
режима/входной частоты. По X — секунды после прогрева, по Y — число завершённых
кадров в окне / длительность окна.
Последнее неполное окно делится на его фактическую длительность, пустые окна дают 0 FPS.
Изменяйте `FPS_BIN_SECONDS` для более подробного или сглаженного отображения.
Эта ячейка читает сохранённые результаты и не запускает inference.


In [ ]:
FPS_BIN_SECONDS = 1.0
if results_path is not None:
    plot_fps_over_time(results_path, bin_seconds=FPS_BIN_SECONDS)
else:
    print("Select a results directory in the table cell above.")


## Задержка (мс) во времени для каждой модели

По X — время завершения кадра в секундах после начала измерения, по Y — задержка
в миллисекундах. Сплошная линия — время предсказания; для синтетического потока
пунктиром дополнительно показана задержка кадра с копированием и очередью.
Все наблюдения сохранены, без фильтрации выбросов
и усреднения. Используются те же сохранённые результаты, что и для FPS.


In [ ]:
if results_path is not None:
    plot_latency_over_time(results_path)
else:
    print("Select a results directory in the table cell above.")
